have all the columns that can have different Capitalization standarized

In [106]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from data_profiling import ProfileReport
%matplotlib inline
pd.set_option('display.max_rows', 10)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

df = pd.read_csv("../data/raw/store-data.csv", encoding="latin-1")

df_copy = df.copy()

df_copy["Customer Name"] = df_copy["Customer Name"].str.strip().str.title()
df_copy["Category"] = df_copy["Category"].str.strip().str.title()
df_copy["Segment"] = df_copy["Segment"].str.strip().str.title()
df_copy["State"] = df_copy["State"].str.strip().str.title()
df_copy["City"] = df_copy["City"].str.strip().str.title()

# df_copy.groupby("Product ID")["Product Name"].transform(lambda x: x.mode().iat[0] if not x.mode().empty else np.nan )

segment_mapping = {
    "Home Ofice" : "Home Office",
    "Consumerr" : "Consumer",
    "Corporrate" : "Corporate"
}
df_copy["Segment"] = df_copy["Segment"].replace(segment_mapping)

display(df_copy)

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2.0,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs, Rounded Back",731.9400,3.0,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters by Universal,14.6200,2.0,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5.0,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2.0,0.20,2.5164
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10059,1281,CA-2016-160815,9/5/2016,9/6/2016,First Class,TR-21325,Toby Ritter,Consumer,United States,Cedar Rapids,Iowa,52402.0,Central,TEC-PH-10003505,Technology,Phones,Geemarc AmpliPOWER60,278.4000,3.0,0.00,80.7360
10060,7699,CA-2017-151799,12/14/2017,12/18/2017,Standard Class,BF-11170,Ben Ferrer,Home Office,United States,Lawrence,Massachusetts,1841.0,East,OFF-ST-10002790,Office Supplies,Storage,Safco Industrial Shelving,73.8500,1.0,0.00,2.2155
10061,8597,CA-2014-111934,5/5/2014,5/7/2014,First Class,GD-14590,Giulietta Dortch,Corporate,United States,Arlington,Virginia,22204.0,South,OFF-PA-10000474,Office Supplies,Paper,Easy-staple paper,35.4400,1.0,0.00,16.6568
10062,4962,CA-2014-156587,3/7/2014,3/8/2014,First Class,AB-10015,NaN,Consumer,United States,Seattle,Washington,98103.0,West,FUR-CH-10004477,Furniture,Chairs,"Global Push Button Manager's Chair, Indigo",48.7120,1.0,0.20,5.4801


fill all the missing values

In [ ]:
df_copy["Postal Code"] = df_copy.groupby("City")["Postal Code"].ffill().bfill()

df_copy["Customer Name"] = df_copy.groupby("Customer ID")["Customer Name"].ffill().bfill()

df_copy["Ship Date"] = df_copy.groupby("Order ID")["Ship Date"].ffill().bfill()

df_copy["Ship Mode"] = df_copy.groupby("Order ID")["Ship Mode"].ffill().bfill()

for mod in ["Sales", "Quantity"]:
    df_copy[mod] = df_copy.groupby(["Product ID", "Profit", "Discount"])[mod].ffill().bfill()



df_copy["Order Date"] = pd.to_datetime(df_copy["Order Date"], format="mixed")
df_copy["Ship Date"] = pd.to_datetime(df_copy["Ship Date"], format="mixed")


df_copy["Shipping Days"] = (df_copy["Ship Date"] - df_copy["Order Date"]).dt.days

estimated = df_copy.groupby("Ship Mode")["Shipping Days"].median()

display(estimated)

days_to_add = pd.to_timedelta(df_copy["Ship Mode"].map(estimated), unit='D')

df_copy["Ship Date"] = np.where(df_copy["Ship Date"] < df_copy["Order Date"],df_copy["Order Date"] + days_to_add, df_copy["Ship Date"])



df_copy["Shipping Days"] = (df_copy["Ship Date"] - df_copy["Order Date"]).dt.days

df_copy["Ship Date"] = np.where(df_copy["Shipping Days"] > 200, df_copy["Order Date"] + days_to_add, df_copy["Ship Date"])

df_copy["Discount"] = df_copy.groupby(["Product ID", "Profit", "Sales"])["Discount"].transform(lambda x: x.median())
df_copy["Quantity"] = df_copy.groupby(["Product ID", "Profit", "Sales"])["Quantity"].transform(lambda x: x.median())
# df_copy["Sales"] = df_copy.groupby(["Product ID", "Profit", "Quantity"])["Sales"].transform(lambda x: x.median())
# df_copy["Quantity"] = df_copy.groupby(["Product ID", "Profit", "Sales"])["Quantity"].transform(lambda x: x.median())

display(df_copy["Discount"])
display(df_copy[df_copy["Quantity"] < 0])


# display(df_copy[df_copy["Ship Date"] < df_copy["Order Date"]])

print("\n\ndataset missing values count \n")
display(df_copy.isna().sum().sort_values(ascending=False))

Ship Mode
First Class       2.0
Same Day          0.0
Second Class      3.0
Standard Class    5.0
Name: Shipping Days, dtype: float64

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Shipping Days
3096,3097,CA-2015-114468,2015-08-23,2015-08-23,Same Day,TD-20995,Tamara Dahlen,Consumer,United States,Bolingbrook,Illinois,60440.0,Central,OFF-AP-10000696,Office Supplies,Appliances,Holmes Odor Grabber,5.768,2.0,1.5,-13.5548,0
3501,3502,CA-2016-108616,2016-09-29,2016-10-03,Standard Class,JK-15730,Joe Kamberova,Consumer,United States,Mobile,Alabama,36608.0,South,OFF-AR-10001149,Office Supplies,Art,"Sanford Colorific Colored Pencils, 12/Box",25.920,9.0,1.5,7.7760,4
4551,4552,US-2014-165862,2014-07-13,2014-07-17,Standard Class,GK-14620,Grace Kelly,Corporate,United States,Los Angeles,California,90049.0,West,FUR-TA-10002855,Furniture,Tables,Bevis Round Conference Table Top & Single Column Base,351.216,3.0,1.5,4.3902,4
6405,6406,CA-2017-154074,2017-08-31,2017-09-02,Second Class,BW-11110,Bart Watters,Corporate,United States,Spokane,Washington,99207.0,West,FUR-CH-10002331,Furniture,Chairs,Hon 4700 Series Mobuis Mid-Back Task Chairs with Adjustable Arms,569.568,2.0,1.5,7.1196,2
6457,6458,CA-2014-110065,2014-08-05,2014-08-11,Standard Class,SP-20650,Stephanie Phelps,Corporate,United States,New York City,New York,10009.0,East,OFF-AR-10004165,Office Supplies,Art,"Binney & Smith inkTank Erasable Pocket Highlighter, Chisel Tip, Yellow",15.960,7.0,1.5,7.0224,6
6683,6684,CA-2014-109134,2014-11-05,2014-11-10,Standard Class,DE-13255,Deanra Eno,Home Office,United States,Los Angeles,California,90008.0,West,FUR-FU-10000320,Furniture,Furnishings,OIC Stacking Trays,20.040,6.0,1.5,8.8176,5


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Shipping Days
439,440,CA-2017-157252,2017-01-20,2017-01-23,Second Class,CV-12805,Cynthia Voltz,Corporate,United States,New York City,New York,10024.0,East,FUR-CH-10003396,Furniture,Chairs,Global Deluxe Steno Chair,207.846,-1.0,0.1,2.3094,3
650,651,CA-2016-160745,2016-12-11,2016-12-16,Second Class,AR-10825,Anthony Rawles,Corporate,United States,Vancouver,Washington,98661.0,West,TEC-AC-10001142,Technology,Accessories,First Data FD10 PIN Pad,316.000,-0.5,0.0,31.6000,5
1481,1482,US-2016-108455,2016-12-02,2016-12-08,Standard Class,MK-18160,Mike Kennedy,Consumer,United States,San Francisco,California,94122.0,West,OFF-ST-10002214,Office Supplies,Storage,X-Rack File for Hanging Folders,33.870,-5.0,0.0,8.8062,6
2302,2303,CA-2016-106558,2016-01-17,2016-01-21,Standard Class,DL-13495,Dionis Lloyd,Corporate,United States,Columbus,Georgia,31907.0,South,TEC-AC-10001142,Technology,Accessories,First Data FD10 PIN Pad,316.000,-0.5,0.0,31.6000,4
6412,6413,CA-2017-151211,2017-08-17,2017-08-23,Standard Class,AH-10120,Adrian Hane,Home Office,United States,Louisville,Kentucky,40214.0,South,OFF-BI-10002735,Office Supplies,Binders,GBC Prestige Therm-A-Bind Covers,102.930,-1.0,0.0,48.3771,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7799,7800,CA-2017-166184,2017-03-24,2017-03-27,First Class,HR-14830,Harold Ryan,Corporate,United States,New York City,New York,10035.0,East,FUR-CH-10003396,Furniture,Chairs,Global Deluxe Steno Chair,207.846,-1.0,0.1,2.3094,3
7894,7895,CA-2017-124744,2017-06-21,2017-06-25,Standard Class,EH-14125,Eugene Hildebrand,Home Office,United States,Wheeling,West Virginia,26003.0,East,OFF-BI-10002852,Office Supplies,Binders,Ibico Standard Transparent Covers,82.400,-5.0,0.0,40.3760,4
8730,8731,CA-2015-103870,2015-12-27,2015-12-31,Standard Class,SP-20860,Sung Pak,Corporate,United States,Murfreesboro,Tennessee,37130.0,South,TEC-AC-10004227,Technology,Accessories,SanDisk Ultra 16 GB MicroSDHC Class 10 Memory Card,72.744,-5.0,0.2,-12.7302,4
9662,9663,CA-2016-160717,2016-06-06,2016-06-11,Standard Class,ME-17320,Maria Etezadi,Home Office,United States,Santa Barbara,California,93101.0,West,TEC-PH-10001760,Technology,Phones,Bose SoundLink Bluetooth Speaker,477.600,-5.0,0.2,161.1900,5




dataset missing values count 



Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
                ..
Sales            0
Quantity         0
Discount         0
Profit           0
Shipping Days    0
Length: 22, dtype: int64

drop all the duplicated

In [108]:
df_copy.drop_duplicates(inplace=True)

profile = ProfileReport(df_copy, title="Profiling Report after cleaning")
profile.to_file("../reports/report_cleaning.html")



Export report to file: 100%|██████████| 1/1 [00:00<00:00, 59.64it/s]
